In [ ]:
import os
import re
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

# Set consistent visualization aesthetics
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

In [ ]:
# Load datasets
data_dir = Path("../data").resolve()
fake_path = data_dir / "Fake.csv"
true_path = data_dir / "True.csv"

df_fake = pd.read_csv(fake_path)
df_true = pd.read_csv(true_path)

df_fake["label"] = 1
df_fake["label_name"] = "Fake"

df_true["label"] = 0
df_true["label_name"] = "Real"

print(f"Fake news shape: {df_fake.shape}")
print(f"Real news shape: {df_true.shape}")

In [ ]:
# Inspect schema and dtypes
print("=== Fake News Schema ===")
print(df_fake.info())
print("\n=== Real News Schema ===")
print(df_true.info())

In [ ]:
# Concatenate and check missing values
df = pd.concat([df_fake, df_true], ignore_index=True)
print(f"Combined total rows: {len(df)}")
print("\nMissing values count:")
print(df.isnull().sum())

In [ ]:
# Class distribution analysis
print("Class distribution count:")
print(df["label_name"].value_counts())
print("\nClass proportions:")
print(df["label_name"].value_counts(normalize=True).round(4))

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x="label_name", data=df, palette=["#ef4444", "#10b981"], ax=ax)
ax.set_title("Class Distribution (Real vs Fake)")
ax.set_xlabel("Label")
ax.set_ylabel("Number of Articles")
plt.tight_layout()
plt.show()

In [ ]:
# Duplicate analysis
exact_dupes = df.duplicated(subset=["title", "text"]).sum()
title_dupes = df.duplicated(subset=["title"]).sum()
text_dupes = df.duplicated(subset=["text"]).sum()

print(f"Exact title+text duplicates: {exact_dupes}")
print(f"Duplicate titles: {title_dupes}")
print(f"Duplicate text bodies: {text_dupes}")

In [ ]:
# Subject and topic analysis
print("Subjects in Fake news:")
print(df_fake["subject"].value_counts())
print("\nSubjects in Real news:")
print(df_true["subject"].value_counts())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
df_fake["subject"].value_counts().plot(kind="bar", color="#ef4444", ax=ax1, title="Fake News Subjects")
df_true["subject"].value_counts().plot(kind="bar", color="#10b981", ax=ax2, title="Real News Subjects")
plt.tight_layout()
plt.show()

In [ ]:
# Data Leakage Investigation: Publisher dateline shortcut
reuters_in_real = df_true["text"].str.contains(r"\(Reuters\)|Reuters", regex=True, na=False).mean()
reuters_in_fake = df_fake["text"].str.contains(r"\(Reuters\)|Reuters", regex=True, na=False).mean()

print(f"Real articles with 'Reuters' dateline: {reuters_in_real * 100:.2f}%")
print(f"Fake articles with 'Reuters' dateline: {reuters_in_fake * 100:.2f}%")

print("\nSample openings in True.csv:")
for s in df_true["text"].head(3):
    print("  ->", s[:90])

print("\nSample openings in Fake.csv:")
for s in df_fake["text"].head(3):
    print("  ->", s[:90])

In [ ]:
# Length distributions
df["title_len"] = df["title"].fillna("").apply(lambda x: len(x.split()))
df["text_len"] = df["text"].fillna("").apply(lambda x: len(x.split()))

print("Title length summary:")
print(df.groupby("label_name")["title_len"].describe())

print("\nText length summary:")
print(df.groupby("label_name")["text_len"].describe())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x="title_len", hue="label_name", bins=30, kde=True, palette=["#ef4444", "#10b981"], ax=ax1)
ax1.set_title("Title Word Count Distribution")
ax1.set_xlim(0, 40)

sns.histplot(data=df[df["text_len"] <= 1500], x="text_len", hue="label_name", bins=40, kde=True, palette=["#ef4444", "#10b981"], ax=ax2)
ax2.set_title("Text Word Count Distribution (<= 1500 words)")
plt.tight_layout()
plt.show()

In [ ]:
# Empty/short text checks
empty_text_count = (df["text"].str.strip().str.len() == 0).sum()
short_text_count = (df["text_len"] < 10).sum()
print(f"Articles with 0 text characters: {empty_text_count}")
print(f"Articles with fewer than 10 words: {short_text_count}")

In [ ]:
# Vocabulary analysis with dateline removed
dateline_pattern = re.compile(r"^[A-Za-z\s,/\.\-]+(?:\([A-Za-z\s,]+\))?\s*[-–—]\s*")
clean_real = df_true["text"].dropna().sample(2000, random_state=42).apply(lambda t: dateline_pattern.sub("", t).strip())
clean_fake = df_fake["text"].dropna().sample(2000, random_state=42).apply(lambda t: dateline_pattern.sub("", t).strip())

vec = CountVectorizer(stop_words="english", max_features=15)
real_counts = vec.fit_transform(clean_real).toarray().sum(axis=0)
real_words = vec.get_feature_names_out()

fake_counts = vec.fit_transform(clean_fake).toarray().sum(axis=0)
fake_words = vec.get_feature_names_out()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.barh(real_words, real_counts, color="#10b981")
ax1.set_title("Top Words in Real News (Dateline Stripped)")
ax2.barh(fake_words, fake_counts, color="#ef4444")
ax2.set_title("Top Words in Fake News")
plt.tight_layout()
plt.show()